### Tools
# Models can request to call tools that perform tasks such as fetching database, searching web or running code. Tools are pairing of :
## 1. A schema, including the name of the tool, a description , and argument definition(often json schema)
## 2. A function or coroutine to execute.

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("why do parratos talk so much?")
response

In [ ]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at given location"""
    return f"The weather at the {location} is cloudy"

model_with_tool = model.bind_tools([get_weather])

In [ ]:
response = model_with_tool.invoke("what is the weather in new york?")
print(response)
for tool_call in response.tool_calls:
    #view tools call made by the model
    print(f"Tool: {tool_call['Name']}")
    print(f"Args: {tool_call['args']}")

### Messages:


In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")

In [ ]:
model.invoke("Please tell me ,what is AI?")

### Text Prompts:

## Text promopts are strings - ideal for straight forward generation tasks where we don't need to retain. conversation history.

In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are a poetry expert!"),
    HumanMessage("Write a poem on AI?")
]

response = model.invoke(messages)
response.content

In [ ]:
system_msg = SystemMessage("You are a helpful coding assitant!")

messages = [
    system_msg,
    HumanMessage("Write a python function to add two numbers")
]

response = model.invoke(messages)
print(response.content)

In [ ]:
## Detailed info to the LLM through system message:
from langchain.messages import SystemMessage, HumanMessage

system_msg = SystemMessage("""
You are a senior python developer with expertise in web frameworks.
Always provide code with proper comments and best practices.
Be concise and to the point in your responses.
                           
""")      

messages = [
    system_msg,
    HumanMessage("Write a python function to add two numbers")
]

response = model.invoke(messages)
print(response.content)

In [ ]:
## Message Meta data:

human_msg = HumanMessage(
    content = "Hello!",
    name = "md_haque",
    id = "1234",
    
)

response = model.invoke([human_msg])
print(response.content)

In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

#create an AI message manually:
ai_msg = AIMessage("I would be happy to help you with the questions!")

#Add to conversation history:

messages = [
    SystemMessage("You are a helpful assistant!"),
    HumanMessage("Can you help me?"),
    ai_msg, #Insert as if it came from the model.
    HumanMessage("What is the capital of France?")
]

response = model.invoke(messages)
print(response.content)

In [ ]:
response.usage_metadata

In [ ]:
from langchain.messages import AIMessage, ToolMessage

#after a model makes a tool call,
#here, we demonstarte manually creating messages for brevity, but in practice, you would create these messages based on the tool calls made by the model.

ai_message= AIMessage(
    content=[],
    tools_calls=[{
        "name": "get_weather",
        "args": {"location": "Bengaluru"},
        "id": "tool_call_1234"
    }]
)

#execute tool and create result:
weather_result = "Cloudy with a chance of rain"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="tool_call_1234"  #must match the id of the tool call in the AI message
)

#continue conversation with tool result:
messages = [
    HumanMessage("What is the weather in Bengaluru?"),
    ai_message, #model's message with tool call
    tool_message, #result of the tool execution
]
response = model.invoke(messages) #model process the result of the tool and generates a response